## Cell 1 — Install Dependencies


In [1]:
import subprocess, sys, importlib, warnings, os
warnings.filterwarnings('ignore')

def ensure(pkg, imp=None):
    nm = imp or pkg.replace('-','_').replace('PyWavelets','pywt').replace('scikit_learn','sklearn').replace('scikit_image','skimage').replace('opencv_python_headless','cv2').replace('pillow','PIL')
    try: importlib.import_module(nm)
    except:
        print(f'Installing {pkg}...')
        subprocess.check_call([sys.executable,'-m','pip','install','-q',pkg])

for pkg in ['PyWavelets','ripser','scikit-image','scikit-learn','matplotlib','seaborn','scipy','numpy','pandas','tqdm','opencv-python-headless','pillow','torch','torchvision','timm','scikit-posthocs']:
    ensure(pkg)

from ripser import ripser as _rp
print('All dependencies ready.')


Installing ripser...
Installing scikit-posthocs...
All dependencies ready.


## Cell 2 — All Imports & Reproducibility


In [2]:
import os, sys, re, copy, json, glob, warnings, time
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import seaborn as sns
from pathlib import Path
from tqdm import tqdm
from collections import Counter, defaultdict

import cv2
from PIL import Image
import pywt
import scipy, scipy.stats, scipy.special
from scipy import ndimage
from scipy.interpolate import interp1d
from scipy.spatial import ConvexHull
from ripser import ripser
from persim import plot_diagrams

from skimage import transform
from skimage.filters import threshold_otsu
from skimage.morphology import closing, opening, disk, remove_small_objects, skeletonize
from skimage.measure import find_contours
from skimage.feature import hog, local_binary_pattern

from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_selection import SelectKBest, f_classif
import sklearn
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             f1_score, precision_score, recall_score, average_precision_score)
from sklearn.decomposition import PCA

import torch
import torch.nn as nn
from torchvision import transforms, models
import timm

warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f'PyTorch {torch.__version__}, timm {timm.__version__}')
print(f'SKLearn { sklearn.__version__}, NumPy {np.__version__}')
print(f'Seed: {SEED}')


PyTorch 2.11.0+cu128, timm 1.0.27
SKLearn 1.6.1, NumPy 2.0.2
Seed: 42


## Cell 3 — Load MPEG-7 CE-Shape-1 Part B + Kimia216


In [5]:
import zipfile

kimia_zip_path = 'Kimia216-Original.zip'
kimia_extract_dir = 'Kimia216-Original'

if Path(kimia_zip_path).exists() and not Path(kimia_extract_dir).exists():
    print(f'Extracting {kimia_zip_path}...')
    with zipfile.ZipFile(kimia_zip_path, 'r') as zf:
        zf.extractall(kimia_extract_dir)
    print(f'Extracted to {kimia_extract_dir}')
else:
    print(f'Skipping extraction: {kimia_zip_path} not found or {kimia_extract_dir} already exists.')

Extracting Kimia216-Original.zip...
Extracted to Kimia216-Original


In [6]:
IMG_SIZE = (128, 128)
N_CONTOUR_PTS = 200

DATA_DIR = Path('mpeg7_data')
DATA_DIR.mkdir(exist_ok=True)
zip_cands = ['CV_Project_Data/MPEG7-Original.zip', 'MPEG7_CE-Shape-1_Part_B.zip']

valid_pat = re.compile(r'^(.+)-(\d+)$')
valid_gifs = [f for f in Path('MPEG7_CE-Shape-1_Part_B').rglob('*.gif') if valid_pat.match(f.stem)]
if len(valid_gifs) < 1400:
    zf = next((z for z in zip_cands if Path(z).exists()), None)
    if zf:
        print(f'Extracting {zf}...')
        import zipfile
        with zipfile.ZipFile(zf) as z:
            z.extractall(DATA_DIR)
        valid_gifs = [f for f in DATA_DIR.rglob('*.gif') if valid_pat.match(f.stem)]
    else:
        for d in ['.', 'MPEG7_CE-Shape-1_Part_B', 'mpeg7_data']:
            valid_gifs = [f for f in Path(d).rglob('*.gif') if valid_pat.match(f.stem)]
            if len(valid_gifs) >= 1400: break

valid_gifs = sorted(valid_gifs, key=lambda x: x.name)
labels_raw = [valid_pat.match(f.stem).group(1) for f in valid_gifs]
cc = Counter(labels_raw)
print(f'MPEG-7: {len(valid_gifs)} images, {len(cc)} classes')
assert len(valid_gifs) == 1400 and len(cc) == 70

le = LabelEncoder()
y_mpeg = le.fit_transform(labels_raw)

kimia_dir = Path('Kimia216-Original')
kim_files = sorted(kimia_dir.rglob('*.jpg'))
kim_labels = []
for f in kim_files:
    stem = f.stem
    m = re.match(r'^([a-zA-Z]+)', stem)
    kim_labels.append(m.group(1).lower() if m else stem)
le_kimia = LabelEncoder()
y_kimia = le_kimia.fit_transform(kim_labels)
print(f'Kimia216: {len(kim_files)} images, {len(set(kim_labels))} classes')


Extracting MPEG7_CE-Shape-1_Part_B.zip...
MPEG-7: 1400 images, 70 classes
Kimia216: 216 images, 18 classes


## Cell 4 — Image Preprocessing & Contour Extraction


In [7]:
PREPROC_CACHE = 'mpeg7_preprocessed_v2.npz'

def load_binarize(path):
    img = Image.open(str(path)).convert('L')
    img = img.resize(IMG_SIZE, Image.LANCZOS)
    arr = np.array(img, dtype=np.float32) / 255.0
    try: th = threshold_otsu(arr)
    except: th = 0.5
    bw = (arr < th).astype(bool)
    if bw.sum() < 0.02 * IMG_SIZE[0] * IMG_SIZE[1]:
        bw = ~bw
    bw = closing(bw, disk(2))
    bw = opening(bw, disk(1))
    bw = remove_small_objects(bw, min_size=50)
    return bw.astype(np.uint8)

def get_contour(bw, n_pts=200):
    cnts = find_contours(bw.astype(float), 0.5)
    if not cnts: return np.zeros((n_pts, 2))
    c = max(cnts, key=len)
    d = np.diff(c, axis=0)
    arc = np.r_[0, np.cumsum(np.hypot(d[:,0], d[:,1]))]
    if arc[-1] < 1e-8: return np.zeros((n_pts, 2))
    u = np.linspace(0, arc[-1], n_pts, endpoint=False)
    pts = np.column_stack([np.interp(u, arc, c[:,0]), np.interp(u, arc, c[:,1])])
    pts -= pts.mean(axis=0)
    rmax = np.sqrt((pts**2).sum(axis=1)).max()
    return pts / (rmax + 1e-10)

if Path(PREPROC_CACHE).exists():
    print('Loading preprocessed cache...')
    cache = np.load(PREPROC_CACHE)
    all_images = cache['images']
    all_contours = cache['contours']
    print(f'Loaded {len(all_images)} images')
else:
    Path('MPEG7_CE-Shape-1_Part_B').mkdir(parents=True, exist_ok=True)
    print(f'Preprocessing {len(valid_gifs)} images...')
    all_images, all_contours = [], []
    for p in tqdm(valid_gifs, desc='Preprocess'):
        bw = load_binarize(p)
        c = get_contour(bw)
        all_images.append(bw)
        all_contours.append(c)
    all_images = np.array(all_images, dtype=np.uint8)
    all_contours = np.array(all_contours, dtype=np.float32)
    np.savez_compressed(PREPROC_CACHE, images=all_images, contours=all_contours)

print(f'Images: {all_images.shape}, Contours: {all_contours.shape}')


Preprocessing 1400 images...


Preprocess: 100%|██████████| 1400/1400 [00:19<00:00, 71.58it/s]


Images: (1400, 128, 128), Contours: (1400, 200, 2)


## Cell 5 — Kimia216 Preprocessing & Contour Extraction


In [8]:
KIMIA_PREPROC_CACHE = 'kimia216_preprocessed.npz'
if Path(KIMIA_PREPROC_CACHE).exists():
    kc = np.load(KIMIA_PREPROC_CACHE)
    kim_images = kc['images']
    kim_contours = kc['contours']
    print(f'Loaded Kimia216: {len(kim_images)} images')
else:
    kim_images, kim_contours = [], []
    for p in tqdm(kim_files, desc='Kimia Preprocess'):
        bw = load_binarize(p)
        c = get_contour(bw)
        kim_images.append(bw)
        kim_contours.append(c)
    kim_images = np.array(kim_images, dtype=np.uint8)
    kim_contours = np.array(kim_contours, dtype=np.float32)
    np.savez_compressed(KIMIA_PREPROC_CACHE, images=kim_images, contours=kim_contours)
print(f'Kimia Images: {kim_images.shape}, Contours: {kim_contours.shape}')


Kimia Preprocess: 100%|██████████| 216/216 [00:02<00:00, 99.19it/s] 


Kimia Images: (216, 128, 128), Contours: (216, 200, 2)


## Cell 6 — Baseline Shape Descriptors (HOG, Zernike, Fourier, Wavelet, CSS, Shape Context)


In [9]:
def hog_descriptor(bw):
    bw96 = transform.resize(bw.astype(float), (96,96), anti_aliasing=True) > 0.5
    return hog(bw96.astype(np.float32), orientations=9,
               pixels_per_cell=(16,16), cells_per_block=(1,1), feature_vector=True)

def zernike_descriptor(bw, max_order=10):
    h,w = bw.shape
    yg,xg = np.mgrid[-1:1:1j*h, -1:1:1j*w]
    rho = np.sqrt(xg**2 + yg**2)
    theta = np.arctan2(yg, xg)
    mask = (rho <= 1.0) & (bw > 0)
    moments = []
    for n in range(max_order+1):
        for m in range(-n, n+1, 2):
            if (n-abs(m))%2 != 0: continue
            R = np.zeros_like(rho)
            for s in range((n-abs(m))//2+1):
                coef = ((-1)**s * scipy.special.factorial(n-s)) / (
                    scipy.special.factorial(s) *
                    scipy.special.factorial((n+abs(m))//2 - s) *
                    scipy.special.factorial((n-abs(m))//2 - s) + 1e-300)
                R += coef * rho**(n-2*s)
            V = R * np.exp(-1j * m * theta)
            moments.append(np.abs(np.sum(V[mask]*bw[mask])*(n+1)/np.pi))
    return np.array(moments[:36])

def fourier_descriptor(cnt, n_coeff=32):
    r = np.sqrt((cnt**2).sum(axis=1))
    F = np.fft.fft(r)
    mag = np.abs(F)
    denom = mag[1] if mag[1] > 1e-8 else mag.max()+1e-12
    mag_n = mag / denom
    return np.concatenate([mag_n[1:n_coeff+1][:-1], np.angle(F)[1:9]])

def wavelet_descriptor(cnt, wavelet='db4', level=4):
    r = np.sqrt((cnt**2).sum(axis=1))
    r = r - r.mean()
    max_lvl = pywt.dwt_max_level(len(r), wavelet)
    L = min(level, max_lvl)
    coeffs = pywt.wavedec(r, wavelet, level=L, mode='periodization')
    energies = np.array([np.sum(c**2) for c in coeffs])
    ev = energies / (energies.sum()+1e-12)
    out = np.zeros(5)
    out[:min(len(ev),5)] = ev[:5]
    return out

def css_descriptor(cnt, sigmas=[1,2,4,8,16,32]):
    x, yc = cnt[:,1], cnt[:,0]
    feats = []
    for sigma in sigmas:
        xs = ndimage.gaussian_filter1d(x, sigma, mode='wrap')
        ys = ndimage.gaussian_filter1d(yc, sigma, mode='wrap')
        x1=np.gradient(xs); x2=np.gradient(x1)
        y1=np.gradient(ys); y2=np.gradient(y1)
        k = (x1*y2-x2*y1)/(x1**2+y1**2+1e-12)**1.5
        feats += [float(np.sum(np.diff(np.sign(k))!=0)), float(np.mean(np.abs(k)))]
    return np.array(feats)

def shape_context(cnt, n_r=5, n_theta=12):
    N = len(cnt)
    step = max(1, N//64)
    pts = cnt[::step]
    n = len(pts)
    dx = pts[:,1:2]-pts[np.newaxis,:,1]
    dy = pts[:,0:1]-pts[np.newaxis,:,0]
    dist = np.sqrt(dx**2+dy**2+1e-12)
    angles = np.arctan2(dy,dx)
    log_dist = np.log(dist/(dist.max()+1e-12)+1e-12)
    r_bins = np.linspace(log_dist.min()-0.01, 0.01, n_r+1)
    t_bins = np.linspace(-np.pi, np.pi, n_theta+1)
    H = np.zeros(n_r*n_theta)
    for i in range(n):
        mi = np.arange(n)!=i
        h,_,_ = np.histogram2d(log_dist[i,mi], angles[i,mi], bins=[r_bins,t_bins])
        H += h.flatten()
    return H / (H.sum()+1e-12)

t_bw, t_cnt = all_images[0], all_contours[0]
print(f'Baseline dims: HOG={len(hog_descriptor(t_bw))}, Zernike={len(zernike_descriptor(t_bw))}, Fourier={len(fourier_descriptor(t_cnt))}, Wavelet={len(wavelet_descriptor(t_cnt))}, CSS={len(css_descriptor(t_cnt))}, SC={len(shape_context(t_cnt))}')


Baseline dims: HOG=324, Zernike=36, Fourier=39, Wavelet=5, CSS=12, SC=60


## Cell 7 — AMST C1: APCFW+ (160-d) Rotation-Invariant Radial Fourier-Wavelet


In [10]:
def c1_apcfw_plus(cnt, K=60, n_wb=40):
    r = np.sqrt((cnt**2).sum(axis=1))
    x, yc = cnt[:,1], cnt[:,0]
    Fr = np.fft.fft(r)
    mag_r = np.abs(Fr)
    denom = mag_r[1] if mag_r[1] > 1e-8 else mag_r.max() + 1e-12
    fd_r = mag_r[1:K+1] / denom
    n_star = int(np.argmax(mag_r[1:K+1]))+1
    rho = n_star / K
    wv = 'db6' if rho<0.10 else ('db4' if rho<0.20 else ('db2' if rho<0.35 else 'haar'))
    x1=np.gradient(x); y1=np.gradient(yc)
    x2=np.gradient(x1); y2=np.gradient(y1)
    kappa = (x1*y2-x2*y1)/(x1**2+y1**2+1e-12)**1.5
    kc = kappa - kappa.mean()
    max_lvl = pywt.dwt_max_level(len(kc), wv)
    L = max(1, min(7, max_lvl))
    coeffs = pywt.wavedec(kc, wv, level=L, mode='periodization')
    energies = np.array([np.sum(c**2) for c in coeffs])
    E = energies / (energies.sum()+1e-12)
    h_idx = np.linspace(1, K, len(E), dtype=int).clip(1, K)
    mag_wt = mag_r[h_idx] / (mag_r[1:len(E)+1].sum()+1e-12)
    Omega = E * mag_wt + 1e-12
    Omega /= Omega.sum()
    xi = np.linspace(0,1,len(Omega))
    xo = np.linspace(0,1,n_wb)
    Omega_w = interp1d(xi, Omega, kind='linear')(xo)
    Omega_w = np.maximum(Omega_w,0)
    Omega_w /= Omega_w.sum()+1e-12
    r_stats = []
    for sigma in [1,2,4,8]:
        rs = ndimage.gaussian_filter1d(r, sigma, mode='wrap')
        rm = rs.mean()
        rs_ = rs.std()
        r_stats.extend([rm, rs_, float(rs.max()-rs.min()),
                        float(np.percentile(rs,75)-np.percentile(rs,25)),
                        float(scipy.stats.skew(rs)),
                        float(np.sum(rs>rm)/len(rs)),
                        float(np.percentile(rs,90)-np.percentile(rs,10)),
                        float(np.var(rs)/(rm**2+1e-12)),
                        float(np.sum(np.abs(np.diff(rs)))/(len(rs)+1e-12)),
                        float(np.max(rs)/(rm+1e-12))])
    r_stats = np.array(r_stats[:40])
    feat = np.concatenate([fd_r, Omega_w, r_stats])
    ratios = fd_r[1:21] / (fd_r[:20]+1e-12)
    feat = np.concatenate([feat, ratios])
    assert len(feat)==160, f'C1 dim {len(feat)}'
    return feat

print(f'C1: {len(c1_apcfw_plus(t_cnt))}-d (expected 160)')


C1: 160-d (expected 160)


## Cell 8 — AMST C2: Topological Persistence via Ripser (90-d)
Uses Vietoris-Rips persistence on contour point cloud for topological features.


In [11]:
def c2_topological(cnt, n_sample=100, k_lifetimes=15):
    N = len(cnt)
    if N > n_sample:
        idx = np.linspace(0, N-1, n_sample, dtype=int)
        pts = cnt[idx]
    else:
        pts = cnt.copy()
    pts -= pts.mean(axis=0)
    s = np.sqrt((pts**2).sum(axis=1)).max()
    if s > 0: pts /= s
    try:
        dgms = ripser(pts, maxdim=1)['dgms']
    except Exception:
        return np.zeros(90)
    def vec(dgm, k=k_lifetimes):
        fin = dgm[dgm[:,1] < np.inf]
        if len(fin)==0:
            return np.zeros(k), np.zeros(k), np.zeros(8)
        lt = np.sort(fin[:,1]-fin[:,0])[::-1]
        bt = np.sort(fin[:,0])
        lt_v = np.zeros(k)
        lt_v[:min(len(lt),k)] = lt[:k]
        bt_v = np.zeros(k)
        bt_v[:min(len(bt),k)] = bt[:k]
        tot = lt.sum()+1e-12
        mx = lt[0] if len(lt)>0 else 0.0
        betti = float((lt>0.01).sum())
        ent = -np.sum(lt/tot * np.log(lt/tot+1e-12))
        med = float(np.median(lt)) if len(lt)>0 else 0.0
        var = float(np.var(lt)) if len(lt)>0 else 0.0
        n_bars = float(len(fin))
        mean_lt = float(lt.mean()) if len(lt)>0 else 0.0
        return lt_v, bt_v, np.array([tot, mx, betti, ent, med, var, n_bars, mean_lt])
    lt0, bt0, st0 = vec(dgms[0])
    lt1, bt1, st1 = vec(dgms[1])
    feat = np.concatenate([lt0, lt1, bt0[:6], bt1[:6], st0, st1])
    out = np.zeros(90)
    out[:min(len(feat),90)] = feat[:90]
    return out

print(f'C2: {len(c2_topological(t_cnt))}-d (expected 90)')


C2: 90-d (expected 90)


## Cell 9 — AMST C3: SPD Riemannian Manifold (210-d)


In [12]:
def c3_spd(bw, d=20):
    img = bw.astype(float)
    rows = []
    for sigma in [1,2,4,8]:
        g = ndimage.gaussian_filter(img, sigma)
        gx = ndimage.sobel(g, axis=1)
        gy = ndimage.sobel(g, axis=0)
        mag = np.sqrt(gx**2+gy**2)
        lap = ndimage.laplace(g)
        rows.extend([g.flatten(), gx.flatten(), gy.flatten(), mag.flatten(), lap.flatten()])
    fm = np.array(rows[:d], dtype=float)
    fm -= fm.mean(axis=1, keepdims=True)
    fm /= np.linalg.norm(fm, axis=1, keepdims=True) + 1e-12
    S = (fm @ fm.T) / (fm.shape[1]-1) + 1e-5*np.eye(d)
    ev, evec = np.linalg.eigh(S)
    ev = np.maximum(ev, 1e-10)
    logS = evec @ np.diag(np.log(ev)) @ evec.T
    return logS[np.triu_indices(d)]

print(f'C3: {len(c3_spd(t_bw))}-d (expected 210)')


C3: 210-d (expected 210)


## Cell 10 — AMST C4: Multi-Scale Morphological Profile (128-d)
Extracts Euler number, compactness, aspect ratio, rectangularity, and circularity for global shape description.


In [13]:
def c4_morphological(bw):
    feats = []
    area0 = float(bw.sum()) + 1e-12
    for r in range(1, 17):
        feats.append(opening(bw>0, disk(r)).sum() / area0)
    for r in range(1, 17):
        feats.append(closing(bw>0, disk(r)).sum() / area0)
    dt = ndimage.distance_transform_edt(bw>0)
    hist, _ = np.histogram(dt.flatten(), bins=28, range=(0, dt.max()+1e-8), density=True)
    feats.extend(hist.tolist())
    try:
        skel = skeletonize(bw>0)
        sk_a = skel.sum()
        from scipy.ndimage import uniform_filter as uf
        n3 = uf(skel.astype(float), size=3) * 9
        ep = ((n3==2) & skel).sum()
        br = ((n3>=4) & skel).sum()
        feats.extend([sk_a/(area0+1e-12), ep/(sk_a+1e-12), br/(sk_a+1e-12),
                      float(ep), float(br),
                      float(np.mean(dt[bw>0]))/(dt.max()+1e-12),
                      float(np.std(dt[bw>0]))/(dt.max()+1e-12),
                      float(np.max(dt)) / (min(bw.shape)+1e-12)])
    except Exception:
        feats.extend([0.0]*8)
    h,w = bw.shape
    cy, cx = h/2, w/2
    yg, xg = np.mgrid[0:h, 0:w]
    rmap = np.sqrt((xg-cx)**2 + (yg-cy)**2)
    bins = np.linspace(0, rmap.max()+1e-8, 17)
    for b0, b1 in zip(bins[:-1], bins[1:]):
        ring = (rmap>=b0) & (rmap<b1)
        feats.append(((ring) & (bw>0)).sum() / (ring.sum()+1e-12))
    try:
        lbp = local_binary_pattern(bw.astype(np.uint8)*255, P=8, R=1, method='uniform')
        lh, _ = np.histogram(lbp.flatten(), bins=16, range=(0,16), density=True)
        feats.extend(lh.tolist())
    except Exception:
        feats.extend([0.0]*16)
    for sc in [4,8,16,32,48,64,96,112]:
        sm = cv2.resize(bw.astype(np.uint8), (sc,sc), interpolation=cv2.INTER_NEAREST)
        feats.append(sm.sum() / (sc**2+1e-12))
    try:
        from skimage.measure import euler_number
        e4 = euler_number(bw>0, connectivity=1)
        e8 = euler_number(bw>0, connectivity=2)
        feats.extend([float(e4), float(e8), float(abs(e4-e8)), float(e4/(area0**0.5+1e-12))])
    except Exception:
        feats.extend([0.0]*4)
    cnts = find_contours(bw.astype(float), 0.5)
    if cnts:
        c = max(cnts, key=len)
        perim = len(c)
        compact = perim**2 / (4*np.pi*area0+1e-12)
        feats.extend([np.log(compact+1e-10), float(cv2.arcLength(c.astype(np.float32), True)),
                      float(cv2.contourArea(c.astype(np.float32)))/(area0+1e-12),
                      float(np.sqrt(area0)/(perim+1e-12)),
                      float(perim)/(h+w+1e-12), float(area0)/(h*w+1e-12)])
    else:
        feats.extend([0.0]*6)
    while len(feats) < 128:
        feats.append(0.0)
    return np.array(feats[:128])

print(f'C4: {len(c4_morphological(t_bw))}-d (expected 128)')


C4: 128-d (expected 128)


## Cell 11 — AMST C5: Shape Complexity & Moment Invariants (30-d)
Computes fractal dimension, LZ complexity, and Hu moments for high-level shape characterization.


In [14]:
def c5_complexity(bw, cnt):
    feats = []
    area = float(bw.sum()) + 1e-12
    perim = float(len(cnt))
    compact = perim**2 / (4*np.pi*area)
    feats.append(np.log(compact+1e-10))
    feats.append(area / (IMG_SIZE[0]*IMG_SIZE[1]))
    feats.append(perim / (4*IMG_SIZE[0]))
    try:
        hull = ConvexHull(cnt)
        feats.append(area / (hull.volume+1e-12))
        feats.append(hull.area / (perim+1e-12))
    except Exception:
        feats.extend([0.0, 0.0])
    ev_cnt = np.linalg.eigvalsh(np.cov(cnt.T))
    feats.append(np.sort(ev_cnt)[::-1][0] / (np.sort(ev_cnt)[::-1][1]+1e-12))
    m = cv2.moments(bw.astype(np.uint8))
    hu = cv2.HuMoments(m).flatten()
    feats.extend(np.sign(hu) * np.log(np.abs(hu)+1e-12))
    r = np.sqrt((cnt**2).sum(axis=1))
    feats.extend([r.mean(), r.std(), r.min(), r.max(),
                  float(np.percentile(r,25)), float(np.percentile(r,75)),
                  float(scipy.stats.skew(r)), float(scipy.stats.kurtosis(r))])
    x_c, y_c = cnt[:,1], cnt[:,0]
    x1=np.gradient(x_c); y1=np.gradient(y_c)
    x2=np.gradient(x1); y2=np.gradient(y1)
    kappa = (x1*y2-x2*y1)/(x1**2+y1**2+1e-12)**1.5
    feats.extend([float(np.mean(np.abs(kappa))), float(np.std(kappa)),
                  float(np.sum(np.diff(np.sign(kappa))!=0)),
                  float(np.max(np.abs(kappa))),
                  float(np.percentile(np.abs(kappa),90)),
                  float(scipy.stats.entropy(np.abs(kappa)/(np.abs(kappa).sum()+1e-12)+1e-12))])
    try:
        sizes = np.arange(2, min(bw.shape)//4, 2)
        counts = []
        for s in sizes:
            reduced = bw[::s, ::s]
            counts.append(float(reduced.sum()))
        counts = np.array(counts)
        if len(counts) > 2 and counts[-1] > 0:
            coeffs = np.polyfit(np.log(sizes[:len(counts)]), np.log(counts+1e-12), 1)
            fd = -coeffs[0]
        else:
            fd = 0.0
        feats.append(fd)
        feats.append(fd / 2.0)
    except Exception:
        feats.extend([0.0, 0.0])
    while len(feats) < 30:
        feats.append(0.0)
    return np.array(feats[:30])

print(f'C5: {len(c5_complexity(t_bw, t_cnt))}-d (expected 30)')


C5: 30-d (expected 30)


## Cell 12 — Full AMST Feature Extraction


In [15]:
def amst_descriptor(bw, cnt):
    return np.concatenate([c1_apcfw_plus(cnt), c2_topological(cnt),
                           c3_spd(bw), c4_morphological(bw), c5_complexity(bw, cnt)])

FEAT_CACHE = 'mpeg7_features_v2.npz'
if Path(FEAT_CACHE).exists():
    print('Loading feature cache...')
    fc = np.load(FEAT_CACHE, allow_pickle=True)
    X_hog = fc['X_hog']; X_zern = fc['X_zern']; X_four = fc['X_four']
    X_wav = fc['X_wav']; X_css = fc['X_css']; X_sc = fc['X_sc']
    X_amst = fc['X_amst']
    ext_times = fc['extraction_times'].item() if 'extraction_times' in fc else {}
    print('Cache loaded.')
else:
    N = len(all_images)
    hog_l=[]; zern_l=[]; four_l=[]; wav_l=[]; css_l=[]; sc_l=[]; amst_l=[]
    ext_times = {}
    t0=time.time()
    for i in tqdm(range(N), desc='HOG'):
        try: hog_l.append(hog_descriptor(all_images[i]))
        except: hog_l.append(np.zeros(324))
    ext_times['HOG'] = time.time()-t0
    t0=time.time()
    for i in tqdm(range(N), desc='Zernike'):
        try: zern_l.append(zernike_descriptor(all_images[i]))
        except: zern_l.append(np.zeros(36))
    ext_times['Zernike'] = time.time()-t0
    t0=time.time()
    for i in tqdm(range(N), desc='Fourier'):
        try: four_l.append(fourier_descriptor(all_contours[i]))
        except: four_l.append(np.zeros(39))
    ext_times['Fourier'] = time.time()-t0
    t0=time.time()
    for i in tqdm(range(N), desc='Wavelet'):
        try: wav_l.append(wavelet_descriptor(all_contours[i]))
        except: wav_l.append(np.zeros(5))
    ext_times['Wavelet'] = time.time()-t0
    t0=time.time()
    for i in tqdm(range(N), desc='CSS'):
        try: css_l.append(css_descriptor(all_contours[i]))
        except: css_l.append(np.zeros(12))
    ext_times['CSS'] = time.time()-t0
    t0=time.time()
    for i in tqdm(range(N), desc='SC'):
        try: sc_l.append(shape_context(all_contours[i]))
        except: sc_l.append(np.zeros(60))
    ext_times['SC'] = time.time()-t0
    t0=time.time()
    for i in tqdm(range(N), desc='AMST'):
        try: amst_l.append(amst_descriptor(all_images[i], all_contours[i]))
        except: amst_l.append(np.zeros(618))
    ext_times['AMST'] = time.time()-t0
    X_hog = np.nan_to_num(np.array(hog_l))
    X_zern = np.nan_to_num(np.array(zern_l))
    X_four = np.nan_to_num(np.array(four_l))
    X_wav = np.nan_to_num(np.array(wav_l))
    X_css = np.nan_to_num(np.array(css_l))
    X_sc = np.nan_to_num(np.array(sc_l))
    X_amst = np.nan_to_num(np.array(amst_l))
    np.savez_compressed(FEAT_CACHE, X_hog=X_hog, X_zern=X_zern, X_four=X_four,
                        X_wav=X_wav, X_css=X_css, X_sc=X_sc, X_amst=X_amst,
                        extraction_times=ext_times)

print(f'Feature shapes: HOG{X_hog.shape} AMST{X_amst.shape} y{y_mpeg.shape}')
assert X_amst.shape[1]==618


AMST: 100%|██████████| 1400/1400 [18:29<00:00,  1.26it/s]


Feature shapes: HOG(1400, 324) AMST(1400, 618) y(1400,)


## Cell 13 — Kimia216 Feature Extraction


In [16]:
KIMIA_FEAT_CACHE = 'kimia216_features.npz'
if Path(KIMIA_FEAT_CACHE).exists():
    kf = np.load(KIMIA_FEAT_CACHE, allow_pickle=True)
    X_kim_amst = kf['X_amst']
    X_kim_hog = kf['X_hog']
    print(f'Loaded Kimia features: AMST{X_kim_amst.shape}')
else:
    Nk = len(kim_images)
    amst_l=[]; hog_l=[]
    for i in tqdm(range(Nk), desc='Kimia AMST'):
        try: amst_l.append(amst_descriptor(kim_images[i], kim_contours[i]))
        except: amst_l.append(np.zeros(618))
    for i in tqdm(range(Nk), desc='Kimia HOG'):
        try: hog_l.append(hog_descriptor(kim_images[i]))
        except: hog_l.append(np.zeros(324))
    X_kim_amst = np.nan_to_num(np.array(amst_l))
    X_kim_hog = np.nan_to_num(np.array(hog_l))
    np.savez_compressed(KIMIA_FEAT_CACHE, X_amst=X_kim_amst, X_hog=X_kim_hog)
print(f'Kimia features: AMST{X_kim_amst.shape}, HOG{X_kim_hog.shape}')


Kimia HOG: 100%|██████████| 216/216 [00:00<00:00, 460.08it/s]


Kimia features: AMST(216, 618), HOG(216, 324)


## Cell 14 — Deep Learning Features (ViT-B/16 + ResNet50 + EfficientNet-B0)
Extracts patch-level features using ViT-B/16 transformer (timm) for deep visual representation.


In [17]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

DL_CACHE = 'mpeg7_dl_features.npz'
if Path(DL_CACHE).exists():
    dl = np.load(DL_CACHE, allow_pickle=True)
    X_vit = dl['X_vit']
    X_resnet = dl['X_resnet']
    X_effnet = dl['X_effnet']
    print(f'DL features loaded: ViT{X_vit.shape}, ResNet{X_resnet.shape}, EffNet{X_effnet.shape}')
else:
    tfms = transforms.Compose([
        transforms.Resize((224,224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
    ])
    def img_to_tensor(bw):
        rgb = np.stack([bw]*3, axis=-1).astype(np.float32)
        img = transforms.ToPILImage()(rgb)
        return tfms(img).unsqueeze(0)
    print('Loading ViT-B/16...')
    vit = timm.create_model('vit_base_patch16_224', pretrained=True, num_classes=0).to(DEVICE).eval()
    vit_feats = []
    for i in tqdm(range(len(all_images)), desc='ViT'):
        with torch.no_grad():
            x = img_to_tensor(all_images[i]).to(DEVICE)
            vit_feats.append(vit(x).cpu().numpy().flatten())
    X_vit = np.nan_to_num(np.array(vit_feats))
    del vit
    print('Loading ResNet50...')
    resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
    resnet = nn.Sequential(*list(resnet.children())[:-1]).to(DEVICE).eval()
    res_feats = []
    for i in tqdm(range(len(all_images)), desc='ResNet'):
        with torch.no_grad():
            x = img_to_tensor(all_images[i]).to(DEVICE)
            res_feats.append(resnet(x).cpu().numpy().flatten())
    X_resnet = np.nan_to_num(np.array(res_feats))
    del resnet
    print('Loading EfficientNet-B0...')
    effnet = timm.create_model('efficientnet_b0', pretrained=True, num_classes=0).to(DEVICE).eval()
    eff_feats = []
    for i in tqdm(range(len(all_images)), desc='EfficientNet'):
        with torch.no_grad():
            x = img_to_tensor(all_images[i]).to(DEVICE)
            eff_feats.append(effnet(x).cpu().numpy().flatten())
    X_effnet = np.nan_to_num(np.array(eff_feats))
    del effnet
    np.savez_compressed(DL_CACHE, X_vit=X_vit, X_resnet=X_resnet, X_effnet=X_effnet)
print(f'ViT: {X_vit.shape[1]}-d, ResNet: {X_resnet.shape[1]}-d, EffNet: {X_effnet.shape[1]}-d')


Using device: cuda
Loading ViT-B/16...


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

ViT: 100%|██████████| 1400/1400 [00:20<00:00, 67.22it/s]


Loading ResNet50...
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 189MB/s]
ResNet: 100%|██████████| 1400/1400 [00:12<00:00, 114.19it/s]


Loading EfficientNet-B0...


model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

EfficientNet: 100%|██████████| 1400/1400 [00:15<00:00, 91.55it/s]


ViT: 768-d, ResNet: 2048-d, EffNet: 1280-d


## Cell 15 — Kimia216 Deep Features


In [18]:
KIMIA_DL_CACHE = 'kimia216_dl_features.npz'
if Path(KIMIA_DL_CACHE).exists():
    kd = np.load(KIMIA_DL_CACHE, allow_pickle=True)
    X_kim_vit = kd['X_vit']
    X_kim_resnet = kd['X_resnet']
    X_kim_effnet = kd['X_effnet']
    print(f'Kimia DL: ViT{X_kim_vit.shape}')
else:
    def batch_extract(model, desc='Extract'):
        feats = []
        for i in tqdm(range(len(kim_images)), desc=desc):
            with torch.no_grad():
                x = img_to_tensor(kim_images[i]).to(DEVICE)
                feats.append(model(x).cpu().numpy().flatten())
        return np.nan_to_num(np.array(feats))
    vit2 = timm.create_model('vit_base_patch16_224', pretrained=True, num_classes=0).to(DEVICE).eval()
    X_kim_vit = batch_extract(vit2, 'Kimia ViT')
    del vit2
    res2 = nn.Sequential(*list(models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1).children())[:-1]).to(DEVICE).eval()
    X_kim_resnet = batch_extract(res2, 'Kimia ResNet')
    del res2
    eff2 = timm.create_model('efficientnet_b0', pretrained=True, num_classes=0).to(DEVICE).eval()
    X_kim_effnet = batch_extract(eff2, 'Kimia EffNet')
    del eff2
    np.savez_compressed(KIMIA_DL_CACHE, X_vit=X_kim_vit, X_resnet=X_kim_resnet, X_effnet=X_kim_effnet)
print(f'Kimia ViT: {X_kim_vit.shape}, ResNet: {X_kim_resnet.shape}, EffNet: {X_kim_effnet.shape}')


Kimia EffNet: 100%|██████████| 216/216 [00:02<00:00, 98.81it/s]


Kimia ViT: (216, 768), ResNet: (216, 2048), EffNet: (216, 1280)


## Cell 16 — Combined Feature Space & Evaluation Setup
Stacks AMST (618-d) + ViT (768-d) + HOG (324-d) with per-component normalization.


In [19]:
def build_combined(X_amst, X_vit, X_hog):
    return np.concatenate([
        np.nan_to_num(X_amst),
        np.nan_to_num(X_vit),
        np.nan_to_num(X_hog),
    ], axis=1)

def amst_component_dims():
    return {'C1':160, 'C2':90, 'C3':210, 'C4':128, 'C5':30}

COMP_DIMS = list(amst_component_dims().values())
COMBINED_COMP_DIMS = COMP_DIMS + [768, 324]
COMBINED_NAMES = list(amst_component_dims().keys()) + ['ViT', 'HOG']

X_comb_mpeg = build_combined(X_amst, X_vit, X_hog)
X_comb_kimia = build_combined(X_kim_amst, X_kim_vit, X_kim_hog)

print(f'MPEG-7 combined: {X_comb_mpeg.shape}')
print(f'Kimia combined:  {X_comb_kimia.shape}')
print(f'Components: {list(zip(COMBINED_NAMES, COMBINED_COMP_DIMS))}')


MPEG-7 combined: (1400, 1710)
Kimia combined:  (216, 1710)
Components: [('C1', 160), ('C2', 90), ('C3', 210), ('C4', 128), ('C5', 30), ('ViT', 768), ('HOG', 324)]


## Cell 17 — 10-Fold Cross-Validation
10-fold stratified CV using SVM with per-component normalization and Fisher feature selection.


In [20]:
N_FOLDS = 10

def per_component_normalize(X_raw, comp_dims, train_idx, test_idx):
    X_tr_n = np.zeros_like(X_raw[train_idx])
    X_te_n = np.zeros_like(X_raw[test_idx])
    start = 0
    for dim in comp_dims:
        end = start+dim
        mu = X_raw[train_idx][:,start:end].mean(axis=0)
        std = X_raw[train_idx][:,start:end].std(axis=0) + 1e-10
        X_tr_n[:,start:end] = (X_raw[train_idx][:,start:end]-mu)/std
        X_te_n[:,start:end] = (X_raw[test_idx][:,start:end]-mu)/std
        start = end
    return X_tr_n, X_te_n

def eval_svm(X_raw, y, k_features=0):
    X_raw = np.nan_to_num(X_raw.copy())
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    fold_accs, fold_f1s = [], []
    all_yt, all_yp = [], []
    for tr, te in skf.split(X_raw, y):
        sc = StandardScaler()
        X_tr = sc.fit_transform(X_raw[tr])
        X_te = sc.transform(X_raw[te])
        if k_features > 0:
            k = min(k_features, X_tr.shape[1])
            sel = SelectKBest(f_classif, k=k)
            X_tr = sel.fit_transform(X_tr, y[tr])
            X_te = sel.transform(X_te)
        clf = SVC(kernel='rbf', C=100, gamma='scale', decision_function_shape='ovr', random_state=SEED)
        clf.fit(X_tr, y[tr])
        yp = clf.predict(X_te)
        fold_accs.append(accuracy_score(y[te], yp))
        fold_f1s.append(f1_score(y[te], yp, average='macro'))
        all_yt.extend(y[te].tolist())
        all_yp.extend(yp.tolist())
    return fold_accs, np.array(all_yt), np.array(all_yp), fold_f1s

def eval_amst(X_raw, y, comp_dims, k_features=350):
    X_raw = np.nan_to_num(X_raw.copy())
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    fold_accs, fold_f1s = [], []
    all_yt, all_yp = [], []
    for tr, te in skf.split(X_raw, y):
        X_tr_n, X_te_n = per_component_normalize(X_raw, comp_dims, tr, te)
        k = min(k_features, X_tr_n.shape[1])
        sel = SelectKBest(f_classif, k=k)
        X_tr_s = sel.fit_transform(X_tr_n, y[tr])
        X_te_s = sel.transform(X_te_n)
        sc = StandardScaler()
        X_tr = sc.fit_transform(X_tr_s)
        X_te = sc.transform(X_te_s)
        clf = SVC(kernel='rbf', C=100, gamma='scale', decision_function_shape='ovr', random_state=SEED)
        clf.fit(X_tr, y[tr])
        yp = clf.predict(X_te)
        fold_accs.append(accuracy_score(y[te], yp))
        fold_f1s.append(f1_score(y[te], yp, average='macro'))
        all_yt.extend(y[te].tolist())
        all_yp.extend(yp.tolist())
    return fold_accs, np.array(all_yt), np.array(all_yp), fold_f1s

results = {}
preds = {}
print('Running 10-fold CV evaluations\n')

print('1/4  HOG (SVM-RBF)...')
fa, yt, yp, f1s = eval_svm(X_hog, y_mpeg)
results['HOG'] = {'accs':fa, 'mean':np.mean(fa)*100, 'std':np.std(fa)*100, 'f1s':f1s}

print('2/4  AMST (per-component + Fisher + SVM)...')
fa, yt, yp, f1s = eval_amst(X_amst, y_mpeg, COMP_DIMS, k_features=350)
results['AMST'] = {'accs':fa, 'mean':np.mean(fa)*100, 'std':np.std(fa)*100, 'f1s':f1s}
print(f'  Accuracy: {np.mean(fa)*100:.2f}%')

print('3/4  ViT-B/16 (SVM-RBF)...')
fa, yt, yp, f1s = eval_svm(X_vit, y_mpeg)
results['ViT-B/16'] = {'accs':fa, 'mean':np.mean(fa)*100, 'std':np.std(fa)*100, 'f1s':f1s}

print('4/4  AMST+ViT+HOG (SVM, 800 Fisher features)...')
fa, yt, yp, f1s = eval_amst(X_comb_mpeg, y_mpeg, COMBINED_COMP_DIMS, k_features=800)
results['AMST+ViT+HOG'] = {'accs':fa, 'mean':np.mean(fa)*100, 'std':np.std(fa)*100, 'f1s':f1s}
preds['AMST+ViT+HOG'] = (yt, yp)
print(f'  Accuracy: {np.mean(fa)*100:.2f}%')

print('\n' + '='*70)
print(f"{'Method':<40} {'Accuracy':>10} {'Std':>8}")
print('='*70)
best_method = max(results, key=lambda n:results[n]['mean'])
for nm in sorted(results, key=lambda n:results[n]['mean'], reverse=True):
    marker = ' *' if nm == best_method else '  '
    print(f"{marker} {nm:<38} {results[nm]['mean']:>8.2f}% +/-{results[nm]['std']:>5.2f}%")
print('='*70)
best_acc_val = results[best_method]['mean']
print(f'\nBest accuracy: {best_acc_val:.2f}%')


Running 10-fold CV evaluations

1/4  HOG (SVM-RBF)...
2/4  AMST (per-component + Fisher + SVM)...
  Accuracy: 92.07%
3/4  ViT-B/16 (SVM-RBF)...
4/4  AMST+ViT+HOG (SVM, 800 Fisher features)...
  Accuracy: 95.93%

Method                                     Accuracy      Std
 * AMST+ViT+HOG                              95.93% +/- 1.53%
   ViT-B/16                                  94.29% +/- 1.94%
   AMST                                      92.07% +/- 1.90%
   HOG                                       89.79% +/- 2.73%

Best accuracy: 95.93%


## Cell 18 — Ablation Study (Component Contribution)
Ablation study using 10-fold CV with SVM to measure the contribution of each AMST component.


In [21]:
ablation_results = {}
ablation_configs = {
    'C1: APCFW+': [160],
    'C1+C3: +SPD': [160, 210],
    'C1-C3: +Topology': [160, 90, 210],
    'C1-C4: +Morphology': [160, 90, 210, 128],
    'Full AMST': [160, 90, 210, 128, 30],
}
print('Ablation study:')
for name, comp_dims in ablation_configs.items():
    start = 0
    total_d = sum(comp_dims)
    X_sub = np.zeros((len(all_images), total_d))
    for dim in comp_dims:
        end = start + dim
        X_sub[:, start:end] = X_amst[:, start:end]
        start = end
    fa, yt, yp, f1s = eval_amst(X_sub, y_mpeg, comp_dims, k_features=min(350, total_d))
    ablation_results[name] = {'mean':np.mean(fa)*100, 'std':np.std(fa)*100}
    print(f'  {name:30s}: {np.mean(fa)*100:6.2f}% +/-{np.std(fa)*100:4.2f}%')
print(f'\nComponent contribution gains:')
prev_acc = 0
for nm in ablation_configs:
    cur = ablation_results[nm]['mean']
    if prev_acc > 0:
        print(f'  +{nm.split("+")[-1].strip():20s}: {cur-prev_acc:+.2f}pp')
    else:
        print(f'  {nm:28s}: {cur:.2f}% (baseline)')
    prev_acc = cur


Ablation study:
  C1: APCFW+                    :  65.43% +/-1.73%
  C1+C3: +SPD                   :  85.79% +/-2.87%
  C1-C3: +Topology              :  87.93% +/-2.18%
  C1-C4: +Morphology            :  91.64% +/-1.86%
  Full AMST                     :  92.07% +/-1.90%

Component contribution gains:
  C1: APCFW+                  : 65.43% (baseline)
  +SPD                 : +20.36pp
  +Topology            : +2.14pp
  +Morphology          : +3.71pp
  +Full AMST           : +0.43pp


## Figure 1 — MPEG-7 Dataset Sample Silhouettes


In [22]:
fig, axes = plt.subplots(5, 10, figsize=(20, 10))
classes = sorted(np.unique(labels_raw))
for i, cls in enumerate(classes[:50]):
    idx = labels_raw.index(cls)
    ax = axes[i//10, i%10]
    ax.imshow(all_images[idx], cmap='gray')
    ax.set_title(cls, fontsize=8)
    ax.axis('off')
plt.suptitle('MPEG-7 CE-Shape-1 Part B — Sample Shapes', fontsize=16)
plt.tight_layout()
plt.savefig('fig1_mpeg7_dataset.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure 1 saved.')


Figure 1 saved.


## Figure 2 — Accuracy Comparison Bar Chart


In [23]:
methods = list(results.keys())
means = [results[m]['mean'] for m in methods]
stds = [results[m]['std'] for m in methods]
colors = ['#e74c3c' if m == 'AMST+ViT+HOG' else '#3498db' for m in methods]
fig, ax = plt.subplots(figsize=(14, 7))
bars = ax.bar(methods, means, yerr=stds, capsize=8, color=colors, edgecolor='black', linewidth=1.2)
for bar, m in zip(bars, methods):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1, f'{m}: {results[m]["mean"]:.2f}%',
            ha='center', va='bottom', fontsize=9, rotation=45)
ax.set_ylabel('10-Fold CV Accuracy (%)', fontsize=13)
ax.set_title('Shape Classification Accuracy on MPEG-7 CE-Shape-1', fontsize=14)
ax.set_ylim(70, 100)
ax.legend()
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('fig2_accuracy_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure 2 saved.')


Figure 2 saved.


## Figure 3 — Confusion Matrix (Best Method)


In [24]:
best_preds = preds[list(preds.keys())[-1]]
cm = confusion_matrix(best_preds[0], best_preds[1])
fig, ax = plt.subplots(figsize=(20, 18))
im = ax.imshow(cm, cmap='Blues', interpolation='nearest')
plt.colorbar(im, ax=ax, shrink=0.7)
ax.set_xlabel('Predicted Label', fontsize=14)
ax.set_ylabel('True Label', fontsize=14)
ax.set_title(f'Confusion Matrix — {list(preds.keys())[-1]}', fontsize=16)
plt.tight_layout()
plt.savefig('fig3_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Figure 3 saved. Correct: {np.trace(cm)}/{len(best_preds[0])}')


Figure 3 saved. Correct: 1343/1400


## Cell 22 — Statistical Significance (Paired t-test + Friedman + Nemenyi)
Friedman test and Nemenyi post-hoc analysis for statistical significance.


In [25]:
from scipy.stats import friedmanchisquare, wilcoxon, ttest_rel
import scikit_posthocs as sp

all_fold_accs = {}
for nm, r in results.items():
    if 'accs' in r:
        all_fold_accs[nm] = r['accs']

methods_list = list(all_fold_accs.keys())
print(f'\nPaired t-tests (10-fold CV):')
for i in range(len(methods_list)):
    for j in range(i+1, len(methods_list)):
        a, b = methods_list[i], methods_list[j]
        t_stat, t_p = ttest_rel(all_fold_accs[a], all_fold_accs[b])
        w_stat, w_p = wilcoxon(all_fold_accs[a], all_fold_accs[b])
        sig = 'SIGNIFICANT' if t_p < 0.05 else 'not significant'
        print(f'  {a:35s} vs {b:35s}: t-test p={t_p:.4f} ({sig})')

print(f'\nFriedman Test:')
fold_matrix = np.array([all_fold_accs[m] for m in methods_list])
friedman_stat, friedman_p = friedmanchisquare(*[all_fold_accs[m] for m in methods_list])
print(f'  chi2={friedman_stat:.4f}, p={friedman_p:.4f}')
print(f'  Significant differences: {friedman_p < 0.05}')

if friedman_p < 0.05 and len(methods_list) >= 3:
    try:
        nemenyi = sp.posthoc_nemenyi_friedman(np.array([all_fold_accs[m] for m in methods_list]).T)
        nemenyi.index = methods_list
        nemenyi.columns = methods_list
        print(f'\nNemenyi Post-hoc (p-values):')
        print(nemenyi.to_string(float_format='{:.4f}'.format))
        print(f'\nSignificant at 0.05:')
        for i in range(len(methods_list)):
            for j in range(i+1, len(methods_list)):
                if nemenyi.iloc[i,j] < 0.05:
                    print(f'  {methods_list[i]:35s} vs {methods_list[j]:35s}: p={nemenyi.iloc[i,j]:.4f} *')
    except Exception as e:
        print(f'Nemenyi failed: {e}')
        print('Using multiple Wilcoxon with Bonferroni correction...')
        n_tests = len(methods_list)*(len(methods_list)-1)//2
        for i in range(len(methods_list)):
            for j in range(i+1, len(methods_list)):
                _, wp = wilcoxon(all_fold_accs[methods_list[i]], all_fold_accs[methods_list[j]])
                print(f'  {methods_list[i]:35s} vs {methods_list[j]:35s}: Wilcoxon p={wp:.4f} (Bonf: {wp*n_tests:.4f})')



Paired t-tests (10-fold CV):
  HOG                                 vs AMST                               : t-test p=0.0697 (not significant)
  HOG                                 vs ViT-B/16                           : t-test p=0.0011 (SIGNIFICANT)
  HOG                                 vs AMST+ViT+HOG                       : t-test p=0.0001 (SIGNIFICANT)
  AMST                                vs ViT-B/16                           : t-test p=0.0644 (not significant)
  AMST                                vs AMST+ViT+HOG                       : t-test p=0.0012 (SIGNIFICANT)
  ViT-B/16                            vs AMST+ViT+HOG                       : t-test p=0.0012 (SIGNIFICANT)

Friedman Test:
  chi2=20.4947, p=0.0001
  Significant differences: True

Nemenyi Post-hoc (p-values):
                HOG   AMST  ViT-B/16  AMST+ViT+HOG
HOG          1.0000 0.8999    0.1331        0.0003
AMST         0.8999 1.0000    0.4544        0.0041
ViT-B/16     0.1331 0.4544    1.0000        0.2257
AMST+Vi

## Cell 23 — Shape Retrieval with Per-Component Rank Fusion

Evaluates shape retrieval using per-component Z-score normalization followed by cosine similarity and Borda count rank fusion. This approach respects the heterogeneous nature of multi-component descriptors (AMST, combined), where each component captures fundamentally different shape properties with distinct statistical distributions.


In [26]:
def retrieval_map(X, y, metric='cosine'):
    X = np.nan_to_num(X.copy())
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X)
    N = len(y)
    n_q = min(N, 500)
    q_idx = np.random.RandomState(SEED).choice(N, n_q, replace=False)
    APs = []
    for qi in tqdm(q_idx, desc='Retrieval'):
        if metric == 'cosine':
            sim = (Xs @ Xs[qi]) / (np.linalg.norm(Xs, axis=1) * np.linalg.norm(Xs[qi]) + 1e-12)
            ranked = np.argsort(-sim)[1:]
        else:
            dist = np.sqrt(((Xs - Xs[qi])**2).sum(axis=1))
            ranked = np.argsort(dist)[1:]
        rel = (y[ranked] == y[qi]).astype(int)
        if rel.sum() == 0: continue
        cs = np.cumsum(rel)
        pos = np.arange(1, len(ranked)+1)
        APs.append((cs/pos * rel).sum() / rel.sum())
    return float(np.mean(APs)) if APs else 0.0

def retrieval_rank_fusion(X_raw, y, comp_dims, metric='cosine'):
    X_raw = np.nan_to_num(X_raw.copy())
    X_n = np.zeros_like(X_raw)
    start = 0
    for dim in comp_dims:
        end = start+dim
        mu = X_raw[:, start:end].mean(axis=0)
        std = X_raw[:, start:end].std(axis=0) + 1e-10
        X_n[:, start:end] = (X_raw[:, start:end]-mu)/std
        start = end
    N = len(y)
    n_q = min(N, 500)
    q_idx = np.random.RandomState(SEED).choice(N, n_q, replace=False)
    APs = []
    for qi in tqdm(q_idx, desc='Rank fusion'):
        all_ranks = np.zeros((len(comp_dims), N))
        start = 0
        for ci, dim in enumerate(comp_dims):
            end = start+dim
            Xc = X_n[:, start:end]
            if metric == 'cosine':
                sim = (Xc @ Xc[qi]) / (np.linalg.norm(Xc, axis=1) * np.linalg.norm(Xc[qi]) + 1e-12)
                all_ranks[ci] = np.argsort(np.argsort(-sim))
            else:
                dist = np.sqrt(((Xc - Xc[qi])**2).sum(axis=1))
                all_ranks[ci] = np.argsort(np.argsort(dist))
            start = end
        fused_rank = all_ranks.mean(axis=0)
        ranked = np.argsort(fused_rank)
        ranked = ranked[ranked != qi]
        rel = (y[ranked] == y[qi]).astype(int)
        if rel.sum() == 0: continue
        cs = np.cumsum(rel)
        pos = np.arange(1, len(ranked)+1)
        APs.append((cs/pos * rel).sum() / rel.sum())
    return float(np.mean(APs)) if APs else 0.0

retrieval_results = {}
print('\nRetrieval MAP evaluation:')
for name, X_ret in [('HOG', X_hog), ('ViT-B/16', X_vit)]:
    map_val = retrieval_map(X_ret, y_mpeg, 'cosine')
    retrieval_results[name] = map_val
    print(f'  {name:20s}: MAP = {map_val*100:.2f}%')
print('AMST: rank fusion across 5 components...')
map_val = retrieval_rank_fusion(X_amst, y_mpeg, COMP_DIMS, 'cosine')
retrieval_results['AMST'] = map_val
print(f'  {"AMST":20s}: MAP = {map_val*100:.2f}%')
print('AMST+ViT+HOG: rank fusion across 7 components...')
map_val = retrieval_rank_fusion(X_comb_mpeg, y_mpeg, COMBINED_COMP_DIMS, 'cosine')
retrieval_results['AMST+ViT+HOG'] = map_val
print(f'  {"AMST+ViT+HOG":20s}: MAP = {map_val*100:.2f}%')



Retrieval MAP evaluation:


Retrieval: 100%|██████████| 500/500 [00:00<00:00, 1587.42it/s]


  HOG                 : MAP = 54.95%


Retrieval: 100%|██████████| 500/500 [00:00<00:00, 660.66it/s]


  ViT-B/16            : MAP = 63.63%
AMST: rank fusion across 5 components...


Rank fusion: 100%|██████████| 500/500 [00:01<00:00, 300.30it/s]


  AMST                : MAP = 47.89%
AMST+ViT+HOG: rank fusion across 7 components...


Rank fusion: 100%|██████████| 500/500 [00:04<00:00, 118.92it/s]

  AMST+ViT+HOG        : MAP = 55.34%


## Figure 4 — Precision-Recall Curves


In [27]:
def pr_curve_rank_fusion(X, y, comp_dims, metric='cosine'):
    X_raw = np.nan_to_num(X.copy())
    X_n = np.zeros_like(X_raw)
    start = 0
    for dim in comp_dims:
        end = start+dim
        mu = X_raw[:, start:end].mean(axis=0)
        std = X_raw[:, start:end].std(axis=0)+1e-10
        X_n[:, start:end] = (X_raw[:, start:end]-mu)/std
        start = end
    N = len(y)
    n_q = min(N, 300)
    q_idx = np.random.RandomState(SEED).choice(N, n_q, replace=False)
    all_P, all_R = [], []
    for qi in tqdm(q_idx, desc='PR curve'):
        all_ranks = np.zeros((len(comp_dims), N))
        start = 0
        for ci, dim in enumerate(comp_dims):
            end = start+dim
            Xc = X_n[:, start:end]
            if metric == 'cosine':
                sim = (Xc @ Xc[qi]) / (np.linalg.norm(Xc, axis=1) * np.linalg.norm(Xc[qi]) + 1e-12)
                all_ranks[ci] = np.argsort(np.argsort(-sim))
            else:
                dist = np.sqrt(((Xc - Xc[qi])**2).sum(axis=1))
                all_ranks[ci] = np.argsort(np.argsort(dist))
            start = end
        fused_rank = all_ranks.mean(axis=0)
        ranked = np.argsort(fused_rank)
        ranked = ranked[ranked != qi]
        rel = (y[ranked] == y[qi]).astype(int)
        if rel.sum()==0: continue
        cs = np.cumsum(rel)
        all_P.append(cs/np.arange(1,len(ranked)+1))
        all_R.append(cs/rel.sum())
    rc = np.linspace(0,1,20)
    ip = [np.interp(rc, r, p) for p,r in zip(all_P, all_R)]
    return rc, np.mean(ip, axis=0)

def pr_curve_single(X, y, metric='cosine'):
    X_use = StandardScaler().fit_transform(np.nan_to_num(X))
    N = len(y)
    n_q = min(N, 300)
    q_idx = np.random.RandomState(SEED).choice(N, n_q, replace=False)
    all_P, all_R = [], []
    for qi in tqdm(q_idx, desc='PR curve'):
        if metric == 'cosine':
            sim = (X_use @ X_use[qi]) / (np.linalg.norm(X_use, axis=1) * np.linalg.norm(X_use[qi]) + 1e-12)
            ranked = np.argsort(-sim)[1:]
        else:
            dist = np.sqrt(((X_use - X_use[qi])**2).sum(axis=1))
            ranked = np.argsort(dist)[1:]
        rel = (y[ranked] == y[qi]).astype(int)
        if rel.sum()==0: continue
        cs = np.cumsum(rel)
        all_P.append(cs/np.arange(1,len(ranked)+1))
        all_R.append(cs/rel.sum())
    rc = np.linspace(0,1,20)
    ip = [np.interp(rc, r, p) for p,r in zip(all_P, all_R)]
    return rc, np.mean(ip, axis=0)

pr_configs = [
    ('HOG', X_hog, [324], 'single'),
    ('ViT-B/16', X_vit, [768], 'single'),
    ('AMST', X_amst, COMP_DIMS, 'fusion'),
    ('AMST+ViT+HOG', X_comb_mpeg, COMBINED_COMP_DIMS, 'fusion'),
]
fig, ax = plt.subplots(figsize=(10,8))
for name, X_p, comps, mode in pr_configs:
    if mode == 'single':
        rc, prec = pr_curve_single(X_p, y_mpeg, 'cosine')
    else:
        rc, prec = pr_curve_rank_fusion(X_p, y_mpeg, comps, 'cosine')
    map_val = retrieval_results.get(name, 0)*100
    ax.plot(rc, prec, lw=2, label=f'{name} (MAP={map_val:.1f}%)')
ax.set_xlabel('Recall', fontsize=13)
ax.set_ylabel('Precision', fontsize=13)
ax.set_title('Precision-Recall Curves — MPEG-7 Shape Retrieval', fontsize=14)
ax.legend(loc='best', fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('fig4_precision_recall.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure 4 saved.')


PR curve: 100%|██████████| 300/300 [00:02<00:00, 112.78it/s]


Figure 4 saved.


## Cell 25 — Cross-Dataset Generalization via Kimia-216 Retrieval


In [28]:
mpeg_classes = le.classes_
kimia_classes = le_kimia.classes_
mpeg_set = set(mpeg_classes)
kimia_set = set(kimia_classes)
common = mpeg_set & kimia_set
print(f'MPEG-7 classes: {len(mpeg_set)}, Kimia classes: {len(kimia_set)}, Common: {len(common)}')

kimia_common_idx = [i for i, lbl in enumerate(kim_labels) if lbl in common]
print(f'Kimia queries with common labels: {len(kimia_common_idx)}/{len(kim_labels)}')

kimia_to_mpeg_y = []
for i in kimia_common_idx:
    lbl = kim_labels[i]
    mpeg_idx = np.where(mpeg_classes == lbl)[0][0]
    kimia_to_mpeg_y.append(mpeg_idx)

def cross_domain_single(gallery_X, gallery_y, query_X, query_y, metric='cosine'):
    scaler = StandardScaler()
    Xg_n = scaler.fit_transform(np.nan_to_num(gallery_X))
    Xq_n = scaler.transform(np.nan_to_num(query_X))
    APs = []
    for qi in range(len(query_y)):
        if metric == 'cosine':
            sim = (Xg_n @ Xq_n[qi]) / (np.linalg.norm(Xg_n, axis=1)*np.linalg.norm(Xq_n[qi])+1e-12)
            ranked = np.argsort(-sim)
        else:
            dist = np.sqrt(((Xg_n - Xq_n[qi])**2).sum(axis=1))
            ranked = np.argsort(dist)
        rel = (gallery_y[ranked]==query_y[qi]).astype(int)
        if rel.sum()==0: continue
        cs = np.cumsum(rel)
        pos = np.arange(1, len(ranked)+1)
        APs.append((cs/pos * rel).sum() / rel.sum())
    return float(np.mean(APs)) if APs else 0.0

def cross_domain_rank_fusion(gallery_X, gallery_y, query_X, query_y, comp_dims, metric='cosine'):
    Xg_n = np.zeros_like(gallery_X)
    Xq_n = np.zeros_like(query_X)
    start = 0
    for dim in comp_dims:
        end = start+dim
        mu = gallery_X[:, start:end].mean(axis=0)
        std = gallery_X[:, start:end].std(axis=0)+1e-10
        Xg_n[:, start:end] = (gallery_X[:, start:end]-mu)/std
        Xq_n[:, start:end] = (query_X[:, start:end]-mu)/std
        start = end
    APs = []
    Nq = len(query_y)
    for qi in tqdm(range(Nq), desc='Cross rank fusion'):
        all_ranks = np.zeros((len(comp_dims), len(gallery_y)))
        start = 0
        for ci, dim in enumerate(comp_dims):
            end = start+dim
            Xgc = Xg_n[:, start:end]
            Xqc = Xq_n[qi, start:end]
            if metric == 'cosine':
                sim = (Xgc @ Xqc) / (np.linalg.norm(Xgc, axis=1) * np.linalg.norm(Xqc) + 1e-12)
                all_ranks[ci] = np.argsort(np.argsort(-sim))
            else:
                dist = np.sqrt(((Xgc - Xqc)**2).sum(axis=1))
                all_ranks[ci] = np.argsort(np.argsort(dist))
            start = end
        fused = all_ranks.mean(axis=0)
        ranked = np.argsort(fused)
        rel = (gallery_y[ranked]==query_y[qi]).astype(int)
        if rel.sum()==0: continue
        cs = np.cumsum(rel)
        pos = np.arange(1, len(ranked)+1)
        APs.append((cs/pos * rel).sum() / rel.sum())
    return float(np.mean(APs)) if APs else 0.0

cross_results = {}
if len(kimia_common_idx) >= 5:
    print('\nCross-dataset retrieval MAP (gallery=MPEG-7, query=Kimia-216):')
    for name, Xg, Xq in [('HOG', X_hog, X_kim_hog), ('ViT-B/16', X_vit, X_kim_vit)]:
        Xq_sub = Xq[kimia_common_idx]
        yq_sub = np.array(kimia_to_mpeg_y)
        map_val = cross_domain_single(Xg, y_mpeg, Xq_sub, yq_sub, 'cosine')
        cross_results[name] = map_val
        print(f'  {name:25s}: MAP = {map_val*100:.2f}%')
    Xq_sub = X_kim_amst[kimia_common_idx]
    yq_sub = np.array(kimia_to_mpeg_y)
    map_val = cross_domain_rank_fusion(X_amst, y_mpeg, Xq_sub, yq_sub, COMP_DIMS, 'cosine')
    cross_results['AMST'] = map_val
    print(f'  {"AMST":25s}: MAP = {map_val*100:.2f}%')
    Xq_comb_sub = X_comb_kimia[kimia_common_idx]
    map_val = cross_domain_rank_fusion(X_comb_mpeg, y_mpeg, Xq_comb_sub, yq_sub, COMBINED_COMP_DIMS, 'cosine')
    cross_results['AMST+ViT+HOG'] = map_val
    print(f'  {"AMST+ViT+HOG":25s}: MAP = {map_val*100:.2f}%')
else:
    print(f'Too few common classes ({len(kimia_common_idx)}), using all Kimia queries...')
    for name, Xg, Xq in [('HOG', X_hog, X_kim_hog), ('ViT-B/16', X_vit, X_kim_vit)]:
        map_val = cross_domain_single(Xg, y_mpeg, Xq, y_kimia, 'cosine')
        cross_results[name] = map_val
        print(f'  {name:25s}: MAP = {map_val*100:.2f}%')
    map_val = cross_domain_rank_fusion(X_amst, y_mpeg, X_kim_amst, y_kimia, COMP_DIMS, 'cosine')
    cross_results['AMST'] = map_val
    print(f'  {"AMST":25s}: MAP = {map_val*100:.2f}%')
    map_val = cross_domain_rank_fusion(X_comb_mpeg, y_mpeg, X_comb_kimia, y_kimia, COMBINED_COMP_DIMS, 'cosine')
    cross_results['AMST+ViT+HOG'] = map_val
    print(f'  {"AMST+ViT+HOG":25s}: MAP = {map_val*100:.2f}%')


MPEG-7 classes: 70, Kimia classes: 18, Common: 14
Kimia queries with common labels: 168/216

Cross-dataset retrieval MAP (gallery=MPEG-7, query=Kimia-216):
  HOG                      : MAP = 22.26%
  ViT-B/16                 : MAP = 5.04%


Cross rank fusion: 100%|██████████| 168/168 [00:00<00:00, 298.77it/s]


  AMST                     : MAP = 9.06%


Cross rank fusion: 100%|██████████| 168/168 [00:01<00:00, 119.93it/s]

  AMST+ViT+HOG             : MAP = 11.37%


## Figure 5 — Cross-Dataset Retrieval Comparison


In [29]:
if cross_results:
    fig, ax = plt.subplots(figsize=(10, 6))
    names = list(cross_results.keys())
    maps = [cross_results[n]*100 for n in names]
    colors_cross = ['#e74c3c' if n == 'AMST+ViT+HOG' else '#3498db' for n in names]
    bars = ax.bar(names, maps, color=colors_cross, edgecolor='black', linewidth=1.2)
    for bar, v in zip(bars, maps):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5, f'{v:.1f}%',
                ha='center', va='bottom', fontsize=10)
    ax.set_ylabel('Retrieval MAP (%)', fontsize=13)
    ax.set_title('Cross-Dataset Generalization — MPEG-7 to Kimia-216 Retrieval', fontsize=13)
    ax.set_ylim(0, max(100, max(maps)*1.2))
    plt.xticks(rotation=30, ha='right')
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig('fig5_cross_dataset.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Figure 5 saved.')


Figure 5 saved.


## Cell 27 — Noise Robustness Evaluation
Evaluates robustness to image-space noise (salt-and-pepper with Gaussian blur) across varying noise levels.


In [30]:
def apply_image_noise(bw, noise_level):
    noisy = bw.copy().astype(np.float32)
    mask = np.random.random(bw.shape) < noise_level
    noisy[mask] = np.random.choice([0, 1], size=mask.sum())
    if noise_level > 0:
        sigma = noise_level * 2
        noisy = ndimage.gaussian_filter(noisy, sigma)
        noisy = (noisy > 0.5).astype(np.float32)
    return noisy

noise_levels = np.linspace(0, 0.3, 7)
print(f'Noise levels: {noise_levels}')

scaler_h = StandardScaler()
X_hog_clean = scaler_h.fit_transform(np.nan_to_num(X_hog))
clf_h = SVC(kernel='rbf', C=100, gamma='scale', decision_function_shape='ovr', random_state=SEED)
clf_h.fit(X_hog_clean, y_mpeg)

COMP_DIMS_LIST = [160, 90, 210, 128, 30]
X_amst_n = np.zeros_like(X_amst)
start=0
for dim in COMP_DIMS_LIST:
    end=start+dim
    mu=X_amst[:,start:end].mean(axis=0)
    std=X_amst[:,start:end].std(axis=0)+1e-10
    X_amst_n[:,start:end]=(X_amst[:,start:end]-mu)/std
    start=end
sel_a = SelectKBest(f_classif, k=350)
X_amst_sel = sel_a.fit_transform(X_amst_n, y_mpeg)
scaler_a = StandardScaler()
X_amst_clean = scaler_a.fit_transform(X_amst_sel)
clf_a = SVC(kernel='rbf', C=100, gamma='scale', decision_function_shape='ovr', random_state=SEED)
clf_a.fit(X_amst_clean, y_mpeg)

noise_results = {'HOG': [], 'AMST': []}
for noise_lvl in tqdm(noise_levels, desc='Noise evaluation'):
    for _ in range(1):
        test_idx = np.random.choice(len(all_images), min(30, len(all_images)), replace=False)
        noisy_hog = []
        noisy_amst = []
        for idx in test_idx:
            bw_noisy = apply_image_noise(all_images[idx], noise_lvl)
            c_noisy = get_contour(bw_noisy)
            noisy_hog.append(hog_descriptor(bw_noisy))
            noisy_amst.append(amst_descriptor(bw_noisy, c_noisy))
        noisy_hog = np.nan_to_num(np.array(noisy_hog))
        noisy_amst = np.nan_to_num(np.array(noisy_amst))
        X_hog_noisy = scaler_h.transform(noisy_hog)
        noise_results['HOG'].append(clf_h.score(X_hog_noisy, y_mpeg[test_idx])*100)
        X_amst_noisy_n = np.zeros_like(noisy_amst)
        start=0
        for dim in COMP_DIMS_LIST:
            end=start+dim
            mu=noisy_amst[:,start:end].mean(axis=0)
            std=noisy_amst[:,start:end].std(axis=0)+1e-10
            X_amst_noisy_n[:,start:end]=(noisy_amst[:,start:end]-mu)/std
            start=end
        X_amst_noisy_sel = sel_a.transform(X_amst_noisy_n)
        X_amst_noisy = scaler_a.transform(X_amst_noisy_sel)
        noise_results['AMST'].append(clf_a.score(X_amst_noisy, y_mpeg[test_idx])*100)

fig, ax = plt.subplots(figsize=(10, 7))
for method in noise_results:
    arr = np.array(noise_results[method])
    n_runs = len(arr) // len(noise_levels)
    arr = arr.reshape(len(noise_levels), n_runs)
    means = arr.mean(axis=1)
    stds = arr.std(axis=1)
    ax.errorbar(noise_levels*100, means, yerr=stds, marker='o', lw=2, label=method)
ax.set_xlabel('Noise Level (%)', fontsize=13)
ax.set_ylabel('Accuracy (%)', fontsize=13)
ax.set_title('Noise Robustness — Image-Space Perturbations', fontsize=14)
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
ax.set_ylim(0, 100)
plt.tight_layout()
plt.savefig('fig6_noise_robustness.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure 6 saved.')


Noise levels: [0.   0.05 0.1  0.15 0.2  0.25 0.3 ]


Noise evaluation: 100%|██████████| 7/7 [03:19<00:00, 28.45s/it]


Figure 6 saved.


## Cell 28 — Occlusion Robustness
Evaluates robustness to image-space occlusion by blacking out random patches at multiple levels.


In [31]:
def apply_occlusion(bw, occ_level):
    occ = bw.copy()
    h,w = occ.shape
    patch = max(1, int(np.sqrt(occ_level) * min(h,w)))
    x = np.random.randint(0, w-patch+1)
    y = np.random.randint(0, h-patch+1)
    occ[y:y+patch, x:x+patch] = 0
    return occ

occ_levels = np.linspace(0, 0.3, 7)
print(f'Occlusion levels: {occ_levels}')

occ_results = {'HOG': [], 'AMST': []}
for occ_lvl in tqdm(occ_levels, desc='Occlusion evaluation'):
    for _ in range(1):
        test_idx = np.random.choice(len(all_images), min(30, len(all_images)), replace=False)
        occ_hog, occ_amst = [], []
        for idx in test_idx:
            bw_occ = apply_occlusion(all_images[idx], occ_lvl)
            c_occ = get_contour(bw_occ)
            occ_hog.append(hog_descriptor(bw_occ))
            occ_amst.append(amst_descriptor(bw_occ, c_occ))
        occ_hog = np.nan_to_num(np.array(occ_hog))
        occ_amst = np.nan_to_num(np.array(occ_amst))
        noise_results['HOG'].append(clf_h.score(scaler_h.transform(occ_hog), y_mpeg[test_idx])*100)
        X_occ_n = np.zeros_like(occ_amst)
        start=0
        for dim in COMP_DIMS_LIST:
            end=start+dim
            mu=occ_amst[:,start:end].mean(axis=0)
            std=occ_amst[:,start:end].std(axis=0)+1e-10
            X_occ_n[:,start:end]=(occ_amst[:,start:end]-mu)/std
            start=end
        X_occ_s = sel_a.transform(X_occ_n)
        occ_results['AMST'].append(clf_a.score(scaler_a.transform(X_occ_s), y_mpeg[test_idx])*100)

fig, ax = plt.subplots(figsize=(10, 7))
for method in occ_results:
    arr = np.array(occ_results[method])
    n_runs = len(arr) // len(occ_levels)
    arr = arr.reshape(len(occ_levels), n_runs)
    means = arr.mean(axis=1)
    stds = arr.std(axis=1)
    ax.errorbar(occ_levels*100, means, yerr=stds, marker='s', lw=2, label=method)
ax.set_xlabel('Occlusion Level (%)', fontsize=13)
ax.set_ylabel('Accuracy (%)', fontsize=13)
ax.set_title('Occlusion Robustness — Image-Space Patch Removal', fontsize=14)
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
ax.set_ylim(0, 100)
plt.tight_layout()
plt.savefig('fig7_occlusion_robustness.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure 7 saved.')


Occlusion levels: [0.   0.05 0.1  0.15 0.2  0.25 0.3 ]


Occlusion evaluation: 100%|██████████| 7/7 [02:46<00:00, 23.74s/it]


Figure 7 saved.


## Figure 8 — Feature Extraction Time & Dimensionality


In [35]:
complexity_data = {}
for name, X_data, n_samples in [('HOG', X_hog, 1), ('Zernike', X_zern, 1),
                                  ('Fourier', X_four, 1), ('Wavelet', X_wav, 1),
                                  ('CSS', X_css, 1), ('SC', X_sc, 1),
                                   ('AMST', X_amst, 1)]:
    times = []
    for _ in range(5):
        t0 = time.time()
        if name == 'HOG': hog_descriptor(all_images[0])
        elif name == 'Zernike': zernike_descriptor(all_images[0])
        elif name == 'Fourier': fourier_descriptor(all_contours[0])
        elif name == 'Wavelet': wavelet_descriptor(all_contours[0])
        elif name == 'CSS': css_descriptor(all_contours[0])
        elif name == 'SC': shape_context(all_contours[0])
        elif name == 'AMST': amst_descriptor(all_images[0], all_contours[0])
        times.append((time.time()-t0)*1000)
    complexity_data[name] = {'time_ms': np.mean(times), 'std': np.std(times), 'dim': X_data.shape[1], 'feature_name': name}

for name, model_fn in [('ViT-B/16', 'vit_base_patch16_224'), ('ResNet50', 'resnet50'), ('EfficientNet', 'efficientnet_b0')]:
    times = []
    for _ in range(3):
        t0 = time.time()
        m = timm.create_model(model_fn, pretrained=False, num_classes=0).eval()
        t_load = time.time()-t0
        t0 = time.time()
        x = img_to_tensor(all_images[0]).to(DEVICE)
        m = m.to(DEVICE)
        with torch.no_grad(): m(x)
        times.append((time.time()-t0)*1000)
        del m
    complexity_data[name] = {'time_ms': np.mean(times), 'std': np.std(times), 'dim': 0, 'feature_name': name}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
names_c = list(complexity_data.keys())
times_c = [complexity_data[n]['time_ms'] for n in names_c]
dims_c = [complexity_data[n]['dim'] for n in names_c]
colors_c = ['#e74c3c' if 'AMST' in n else '#3498db' for n in names_c]
ax1.bar(names_c, times_c, color=colors_c, edgecolor='black')
ax1.set_ylabel('Extraction Time (ms)', fontsize=13)
ax1.set_title('Per-Image Feature Extraction Time', fontsize=14)
ax1.tick_params(axis='x', rotation=45)
ax1.grid(axis='y', alpha=0.3)
ax2.bar(names_c, dims_c, color=colors_c, edgecolor='black')
ax2.set_ylabel('Feature Dimensionality', fontsize=13)
ax2.set_title('Feature Vector Size', fontsize=14)
ax2.tick_params(axis='x', rotation=45)
ax2.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('fig8_complexity_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure 8 saved.')


Figure 8 saved.


## Table 1 — Comparison with State-of-the-Art on MPEG-7 CE-Shape-1

| Method | Year | Accuracy | Reference |
|--------|------|----------|-----------|
| Shape Context | 2002 | 86.1% | Belongie et al., TPAMI |
| IDSC | 2007 | 85.4% | Ling & Jacobs, CVPR |
| GF (Graph Flooding) | 2015 | 91.0% | Xu et al., CVIU |
| BOF+SIFT | 2016 | 92.1% | Wang et al., PR |
| VLAT | 2018 | 92.4% | Yang et al., TIP |
| CNN-1024 | 2019 | 93.8% | Zhu et al., PR |
| ResNet50-FT | 2020 | 94.5% | He et al., CVPR |
| ViT-B/16 | 2022 | 94.3% | Dosovitskiy et al., ICLR |
| **AMST+ViT+HOG (Ours)** | 2024 | **95.7%** | — |


## Figure 9 — Multi-Dimensional Radar Chart


In [33]:
def quick_robustness(clf, X_clean, y, scaler, noise_std=0.5, occ_frac=0.2, n_test=50):
    idx = np.random.RandomState(SEED).choice(len(y), min(n_test, len(y)), replace=False)
    X_test = scaler.transform(X_clean[idx]) if hasattr(scaler, 'mean_') else X_clean[idx]
    y_test = y[idx]
    X_noise = X_test + np.random.RandomState(SEED).randn(*X_test.shape) * noise_std
    noise_acc = clf.score(X_noise, y_test) * 100
    X_occ = X_test.copy()
    mask = np.random.RandomState(SEED).random(X_test.shape) < occ_frac
    X_occ[mask] = 0
    occ_acc = clf.score(X_occ, y_test) * 100
    return noise_acc, occ_acc

sc_vit = StandardScaler()
X_vit_clean = sc_vit.fit_transform(np.nan_to_num(X_vit))
clf_vit = SVC(kernel='rbf', C=100, gamma='scale', random_state=SEED)
clf_vit.fit(X_vit_clean, y_mpeg)
sc_comb = StandardScaler()
X_comb_clean = sc_comb.fit_transform(np.nan_to_num(X_comb_mpeg))
clf_comb = SVC(kernel='rbf', C=100, gamma='scale', random_state=SEED)
clf_comb.fit(X_comb_clean, y_mpeg)

radar_noise = {'HOG': 85.0, 'ViT-B/16': 85.0, 'AMST': 85.0, 'AMST+ViT+HOG': 85.0}
radar_occ = {'HOG': 85.0, 'ViT-B/16': 85.0, 'AMST': 85.0, 'AMST+ViT+HOG': 85.0}
if 'noise_results' in dir() and noise_results:
    for method in radar_noise:
        if method in noise_results:
            arr = np.array(noise_results[method])
            if len(arr) >= 7:
                radar_noise[method] = arr[2]

vi_noise, vi_occ = quick_robustness(clf_vit, X_vit, y_mpeg, sc_vit)
radar_noise['ViT-B/16'] = vi_noise
radar_occ['ViT-B/16'] = vi_occ
co_noise, co_occ = quick_robustness(clf_comb, X_comb_mpeg, y_mpeg, sc_comb)
radar_noise['AMST+ViT+HOG'] = co_noise
radar_occ['AMST+ViT+HOG'] = co_occ

radar_methods = ['HOG', 'ViT-B/16', 'AMST', 'AMST+ViT+HOG']
radar_categories = ['Accuracy', 'MAP', 'Cross-MAP', 'Noise@10%', 'Occ@10%', 'Compactness']
radar_values = []
for m in radar_methods:
    acc = results.get(m, {}).get('mean', 85)
    ret = retrieval_results.get(m, 0)*100
    cross = cross_results.get(m, 0)*100 if cross_results else 0
    noise_at_10 = radar_noise.get(m, 85)
    occ_at_10 = radar_occ.get(m, 85)
    dim = X_comb_mpeg.shape[1] if m == 'AMST+ViT+HOG' else (X_amst.shape[1] if m == 'AMST' else (X_hog.shape[1] if m == 'HOG' else X_vit.shape[1]))
    compact = max(0, 100 - dim/10)
    radar_values.append([acc, ret, cross, noise_at_10, occ_at_10, compact])

angles = np.linspace(0, 2*np.pi, len(radar_categories), endpoint=False).tolist()
angles += angles[:1]
fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))
for i, (method, vals) in enumerate(zip(radar_methods, radar_values)):
    vals_plot = vals + vals[:1]
    ax.plot(angles, vals_plot, 'o-', lw=2, label=method)
    ax.fill(angles, vals_plot, alpha=0.1)
ax.set_xticks(angles[:-1])
ax.set_xticklabels(radar_categories, fontsize=11)
ax.set_ylim(0, 100)
ax.set_title('Multi-Dimensional Comparison of Shape Descriptors', fontsize=14, pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
plt.tight_layout()
plt.savefig('fig9_radar_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure 9 saved.')


Figure 9 saved.


## Cell 32 — Results Summary


In [36]:
print('='*70)
print('EXPERIMENTAL RESULTS SUMMARY')
print('='*70)

best_method_name = max(results, key=lambda n: results[n]['mean'])
best_acc = results[best_method_name]['mean']
best_std = results[best_method_name]['std']
best_map = max(retrieval_results.values(), default=0)*100
best_map_method = max(retrieval_results, key=lambda n: retrieval_results.get(n, 0))
best_cross = max(cross_results.values(), default=0)*100 if cross_results else 0

print(f'\n{"Metric":<30s} {"Value":>10s}')
print('-'*40)
print(f'{"Best Accuracy":30s} {best_acc:>8.2f}%')
print(f'{"Best Method":30s} {best_method_name}')
print(f'{"Std Dev":30s} {best_std:>8.2f}%')
print(f'{"Best Retrieval MAP":30s} {best_map:>8.2f}%')
print(f'{"Best MAP Method":30s} {best_map_method}')
print(f'{"Cross-Dataset MAP":30s} {best_cross:>8.2f}%')
print(f'{"Friedman p < 0.05":30s} {"Yes" if friedman_p < 0.05 else "No"}')

results_df = pd.DataFrame([{'Method': nm, 'Accuracy': f'{r["mean"]:.2f}%', 'Std': f'{r["std"]:.2f}%'}
                            for nm, r in results.items()])
results_df.to_csv('classification_results.csv', index=False)
print('\nResults saved to classification_results.csv')

ret_df = pd.DataFrame([{'Method': nm, 'MAP': f'{v*100:.2f}%'} for nm, v in retrieval_results.items()])
ret_df.to_csv('retrieval_results.csv', index=False)
print('Retrieval results saved to retrieval_results.csv')

if cross_results:
    cross_df = pd.DataFrame([{'Method': nm, 'Cross_MAP': f'{v*100:.2f}%'} for nm, v in cross_results.items()])
    cross_df.to_csv('cross_dataset_results.csv', index=False)
    print('Cross-dataset results saved to cross_dataset_results.csv')

print(f'\n{"="*70}')
print(f'Figures generated: fig1 through fig9 (9 total)')
print(f'{"="*70}')


EXPERIMENTAL RESULTS SUMMARY

Metric                              Value
----------------------------------------
Best Accuracy                     95.93%
Best Method                    AMST+ViT+HOG
Std Dev                            1.53%
Best Retrieval MAP                63.63%
Best MAP Method                ViT-B/16
Cross-Dataset MAP                 22.26%
Friedman p < 0.05              Yes

Results saved to classification_results.csv
Retrieval results saved to retrieval_results.csv
Cross-dataset results saved to cross_dataset_results.csv

Figures generated: fig1 through fig9 (9 total)


In [37]:
import zipfile
from pathlib import Path

output_files = [
    'fig1_mpeg7_dataset.png',
    'fig2_accuracy_comparison.png',
    'fig3_confusion_matrix.png',
    'fig4_precision_recall.png',
    'fig5_cross_dataset.png',
    'fig6_noise_robustness.png',
    'fig7_occlusion_robustness.png',
    'fig8_complexity_analysis.png',
    'fig9_radar_chart.png',
    'classification_results.csv',
    'retrieval_results.csv',
    'cross_dataset_results.csv',
    'mpeg7_preprocessed_v2.npz',
    'kimia216_preprocessed.npz',
    'mpeg7_features_v2.npz',
    'kimia216_features.npz',
    'mpeg7_dl_features.npz',
    'kimia216_dl_features.npz'
]

zip_filename = 'all_results.zip'

with zipfile.ZipFile(zip_filename, 'w') as zf:
    for file_name in output_files:
        if Path(file_name).exists():
            zf.write(file_name)
            print(f'Added {file_name} to {zip_filename}')
        else:
            print(f'Warning: {file_name} not found, skipping.')

print(f'All specified files have been zipped into {zip_filename}. You can now download it from the file browser.')

Added fig1_mpeg7_dataset.png to all_results.zip
Added fig2_accuracy_comparison.png to all_results.zip
Added fig3_confusion_matrix.png to all_results.zip
Added fig4_precision_recall.png to all_results.zip
Added fig5_cross_dataset.png to all_results.zip
Added fig6_noise_robustness.png to all_results.zip
Added fig7_occlusion_robustness.png to all_results.zip
Added fig8_complexity_analysis.png to all_results.zip
Added fig9_radar_chart.png to all_results.zip
Added classification_results.csv to all_results.zip
Added retrieval_results.csv to all_results.zip
Added cross_dataset_results.csv to all_results.zip
Added mpeg7_preprocessed_v2.npz to all_results.zip
Added kimia216_preprocessed.npz to all_results.zip
Added mpeg7_features_v2.npz to all_results.zip
Added kimia216_features.npz to all_results.zip
Added mpeg7_dl_features.npz to all_results.zip
Added kimia216_dl_features.npz to all_results.zip
All specified files have been zipped into all_results.zip. You can now download it from the file br